In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/competitions/leap-atmospheric-physics-ai-climsim/'):
    print('hi')
    for filename in filenames:
        
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

hi
/kaggle/input/competitions/leap-atmospheric-physics-ai-climsim/sample_submission.csv
/kaggle/input/competitions/leap-atmospheric-physics-ai-climsim/test_old.csv
/kaggle/input/competitions/leap-atmospheric-physics-ai-climsim/train.csv
/kaggle/input/competitions/leap-atmospheric-physics-ai-climsim/test.csv
/kaggle/input/competitions/leap-atmospheric-physics-ai-climsim/sample_submission_old.csv


In [2]:
import os
import polars as pl
import numpy as np

In [3]:
train_df = pl.scan_csv('/kaggle/input/competitions/leap-atmospheric-physics-ai-climsim/train.csv',infer_schema_length=1000)

test_df = pl.scan_csv('/kaggle/input/competitions/leap-atmospheric-physics-ai-climsim/test.csv',infer_schema_length=1000)

In [4]:
print(train_df.collect_schema().names()) #Gives me the column names of the csv file
print("%"*120)
print(f"train_df : number of rows: {train_df.select(pl.len()).collect()}, number of columns: {len(train_df.collect_schema().names())}")

['sample_id', 'state_t_0', 'state_t_1', 'state_t_2', 'state_t_3', 'state_t_4', 'state_t_5', 'state_t_6', 'state_t_7', 'state_t_8', 'state_t_9', 'state_t_10', 'state_t_11', 'state_t_12', 'state_t_13', 'state_t_14', 'state_t_15', 'state_t_16', 'state_t_17', 'state_t_18', 'state_t_19', 'state_t_20', 'state_t_21', 'state_t_22', 'state_t_23', 'state_t_24', 'state_t_25', 'state_t_26', 'state_t_27', 'state_t_28', 'state_t_29', 'state_t_30', 'state_t_31', 'state_t_32', 'state_t_33', 'state_t_34', 'state_t_35', 'state_t_36', 'state_t_37', 'state_t_38', 'state_t_39', 'state_t_40', 'state_t_41', 'state_t_42', 'state_t_43', 'state_t_44', 'state_t_45', 'state_t_46', 'state_t_47', 'state_t_48', 'state_t_49', 'state_t_50', 'state_t_51', 'state_t_52', 'state_t_53', 'state_t_54', 'state_t_55', 'state_t_56', 'state_t_57', 'state_t_58', 'state_t_59', 'state_q0001_0', 'state_q0001_1', 'state_q0001_2', 'state_q0001_3', 'state_q0001_4', 'state_q0001_5', 'state_q0001_6', 'state_q0001_7', 'state_q0001_8', 'st

# Obtaining statistics of each variable
**This section will compute the statistics of each variable. Statistics are mean, standard deviation, min, max values. Additionally, the variables will be separated into variables that have vertical level and variables without vertical level. For the variables with vertical levels, the statistics will be perform for the whole vertical level and also at each level.**

In [4]:
def vertical_NO_vertical(cols_name_in,col_patterns):
    '''
    This function separates columns names into two categories, those with vertical dimensions and those without vertical
    dimension (scalar)
    Inputs:
    cols_name_in: list, total number column names
    col_patterns: list,pattern with the start name of the vertical variables
    Ouptus:
    X_col_series_out: list, columns names with vertical levels 
    X_col_series_NOT_out: list, columns names without vertical levels
    X_col_series_indices_out: array, columns indices with variables that have vertical levels 
    X_col_series_NOT_indices_out: array, columns indices with variables that DON'T have vertical levels 
    '''
    X_col_series_out = [x for x in cols_name_in if np.mean([x.startswith(y) for y in col_patterns])>0] #it will be true
                # only for when the column starts with one of the col_name_parent and the mean will be larger than 0

    X_col_series_NOT_out  = [x for  x in cols_name_in if x not in X_col_series_out ] #Select the columns that don't have vertical dimension
    
    #Checks if the number of columns with and without vertical levels is the same as the initial (combined) number of columns
    print(f"Sum of vertical and not vertical variables equal to initial number of columns: --> {len(X_col_series_out )+len(X_col_series_NOT_out ) == len(cols_name_in)}")
    
    #----------- Obtaining the inidices of the columns for each varaible clase ------------------
    #---------- variables with vertical dimensions
    X_col_series_indices_out = np.asarray([np.where(np.asarray(cols_name_in) == x)[0][0] for x in X_col_series_out ])
    #---------- variables withOUT vertical dimensions
    X_col_series_NOT_indices_out = np.asarray([np.where(np.asarray(cols_name_in) == x)[0][0] for x in X_col_series_NOT_out ])

    return(X_col_series_out, X_col_series_NOT_out, X_col_series_indices_out, X_col_series_NOT_indices_out)

In [5]:
FEAT_COLS = train_df.collect_schema().names()[1:557] #Input features (columns names)
TARGET_COLS = train_df.collect_schema().names()[557:] #Targets (columns names)
##
#Selecting the variables that have vertical levels
col_name_parent = ['state_t', 'state_q0001', 'state_q0002', 
                   'state_q0003', 'state_u', 'state_v',
                   'pbuf_ozone', 'pbuf_CH4', 'pbuf_N2O'] # These are the basename of the variables with vertical levels
## Obtaining feature columns names and indices for variables with vertical and not vertical levels
(X_col_series,
    X_col_series_NOT,
    X_col_series_indices,
    X_col_series_NOT_indices,
) = vertical_NO_vertical(cols_name_in = FEAT_COLS, col_patterns = col_name_parent)

## ---------- Dividing target variables into variables with vertical leves and variables without vertical levels
y_name_parent = ['ptend_t', 'ptend_q0001', 'ptend_q0002', 'ptend_q0003', 'ptend_u', 'ptend_v']
(y_col_series,
    y_col_series_NOT,
    y_col_series_indices,
    y_col_series_NOT_indices,
) = vertical_NO_vertical(cols_name_in = TARGET_COLS, col_patterns = y_name_parent)

Sum of vertical and not vertical variables equal to initial number of columns: --> True
Sum of vertical and not vertical variables equal to initial number of columns: --> True


**Up to this point I have the column indices for the "train_df" dataframe for the 2D variables and 3D variables**

In [6]:
# def get_statistics(data_in, cols_in):
#     '''
#     It selects the columns in "cols_in" and perform basic statistic operations
#     on the dataset: mean, std, min, max. It uses polars library.
#     Ouput: 
#     returns the statistics for the time and space dimensions (Global statistics)
#     '''
#     stats = (data_in.lazy().select(cols_in) #Selects only the columns  in cols_in
#             .select([ #Performs statistics on the selected columns
#                 pl.all().mean().name.suffix("_mean"),
#                 pl.all().std().name.suffix("_std"),
#                 pl.all().min().name.suffix("_min"),
#                 pl.all().max().name.suffix("_max")
#             ]).collect(engine="streaming"))
#     #--------- Defining outputs ------------
#     mean_out = [stats[f"{col}_mean"][0] for col in cols_in]
#     std_out  = [stats[f"{col}_std"][0]  for col in cols_in]
#     min_out  = [stats[f"{col}_min"][0]  for col in cols_in]
#     max_out  = [stats[f"{col}_max"][0]  for col in cols_in]
#     return(mean_out, std_out, min_out, max_out )
#--------------------------------------------------------------------------------
from tqdm.notebook import tqdm  # notebook-friendly progress bar
import polars as pl
import numpy as np

def get_statistics(data_in, cols_in, n_rows_total, batch_size=500_000,row_offset=0):
    '''
    Computes mean, std, min, max over cols_in with a progress bar.
    Splits the data into batches of batch_size rows and aggregates.
    '''
    n_batches = int(np.ceil(n_rows_total / batch_size))
    
    # Accumulators for online statistics
    batch_means = []
    batch_stds  = []
    batch_mins  = []
    batch_maxs  = []
    batch_counts = []

    source = data_in.lazy().select(cols_in)
    #if row_offset > 0:
    source = source.slice(row_offset,n_rows_total)   # skip first half, then read sequentially

    batches = source.collect_batches(chunk_size=batch_size)

    with tqdm(total=n_batches, desc="Computing statistics", unit="batch") as pbar:
        # for i in range(n_batches):
        for batch_df in batches:
            n = len(batch_df)
            if n == 0:
                continue
            #start = row_offset + i * batch_size
            
            # stats = (
            #     data_in.lazy()
            #     .select(cols_in)
            #     .slice(start, batch_size)          # explicit row window
            #     .select([
            #         pl.all().mean().name.suffix("_mean"),
            #         pl.all().std() .name.suffix("_std"),
            #         pl.all().min() .name.suffix("_min"),
            #         pl.all().max() .name.suffix("_max"),
            #         pl.lit(batch_size).alias("_count")
            #     ])
            #     .collect(engine="streaming")
            # )
            stats = (batch_df.select([
                    pl.all().mean().name.suffix("_mean"),
                    pl.all().std() .name.suffix("_std"),
                    pl.all().min() .name.suffix("_min"),
                    pl.all().max() .name.suffix("_max"),
                    pl.lit(n).alias("_count")
                ])
            )
            
            batch_means .append([stats[f"{col}_mean"][0] for col in cols_in])
            batch_stds  .append([stats[f"{col}_std"][0]  for col in cols_in])
            batch_mins  .append([stats[f"{col}_min"][0]  for col in cols_in])
            batch_maxs  .append([stats[f"{col}_max"][0]  for col in cols_in])
            batch_counts.append(stats["_count"][0])

            # pbar.set_postfix({
            #     "rows processed": f"{min((i+1)*batch_size, n_rows_total):,}",
            #     "remaining"     : f"{max(n_rows_total - (i+1)*batch_size, 0):,}"
            # })
            pbar.set_postfix({
                "rows": f"{sum(batch_counts):,}",
                "RAM" : f"{psutil.Process().memory_info().rss/1e9:.1f} GB"
            })
            pbar.update(1)

    # --- Combine batches ---
    # min/max: just take element-wise min/max across batches
    min_out = np.min (batch_mins,  axis=0).tolist()
    max_out = np.max (batch_maxs,  axis=0).tolist()

    # mean: weighted average across batches
    counts  = np.array(batch_counts)             # shape (n_batches,)
    means   = np.array(batch_means)              # shape (n_batches, n_cols)
    mean_out = (means * counts[:, None]).sum(axis=0) / counts.sum()

    # std: combine via sum of squares (correct global std)
    stds    = np.array(batch_stds)               # shape (n_batches, n_cols)
    # variance = mean of (var_i + mean_i^2) - global_mean^2
    vars_   = stds**2 + means**2                 # E[X^2] per batch
    global_ex2 = (vars_ * counts[:, None]).sum(axis=0) / counts.sum()
    std_out  = np.sqrt(np.clip(global_ex2 - mean_out**2, 0, None)).tolist()
    

    return mean_out, std_out, min_out, max_out

#---------------------------------------------------------------------------------

def get_statistics_vertical_var(data_in, cols_in, n_rows_total, batch_size=500_000,row_offset=0):
    '''
    Given the structure of the data, each vertical variable is splitted into
    sub-variables for each vertical level. For example state_01 has 60 vertical
    levels. In the dataframe 60 columns will be found with the names:
    state_01_0, state_01_1,..., state_01_59. This function collapses all these
    columns into one to get the mean value per variable in location (lat,lon) and 
    vertical level. For example if "data_in" shape is: (10,540), 540 columns are 
    the product of: "number_var_vertical"*60, so doing 540/60 will give the
    number of variables with vertical dimension ("number_var_vertical"). That
    part corresponds to: np.shape(train_ds_dummy.select(X_col_series))[1]//60).
    Then the other dimension is 60 as each variable has 60 vertical levels. The
    final dimension will be (10,9,60).
    
    Output:
    Statistic for each variable with vertical level. The statistics are perform
    over all vertical levels at the same time.
    '''
    n_levels = 60
    n_vars   = len(cols_in) // n_levels

    var_groups = {
        f"var_{i}": cols_in[i*n_levels:(i+1)*n_levels]
        for i in range(n_vars)
    }

    # Accumulators — one list per variable
    all_means  = [[] for _ in range(n_vars)]
    all_stds   = [[] for _ in range(n_vars)]
    all_mins   = [[] for _ in range(n_vars)]
    all_maxs   = [[] for _ in range(n_vars)]
    all_counts = [[] for _ in range(n_vars)]

    # Single pass — read ALL columns at once, process per variable inside loop
    source  = (
        data_in.lazy()
        .select(cols_in)               # all columns in one read
        .slice(row_offset, n_rows_total)
    )
    batches = source.collect_batches(chunk_size=batch_size)

    with tqdm(total=n_rows_total//batch_size + 1, desc="Reading CSV", unit="batch") as pbar:
        for batch_df in batches:
            if len(batch_df) == 0:
                continue

            # Process all variables on this batch while it's in RAM
            for i, (var_name, level_cols) in enumerate(var_groups.items()):
                stats = (
                    batch_df.select(level_cols)   # slice columns in RAM — free
                    .unpivot(on=level_cols, variable_name="level", value_name="value")
                    .select([
                        pl.col("value").mean().alias("mean"),
                        pl.col("value").std() .alias("std"),
                        pl.col("value").min() .alias("min"),
                        pl.col("value").max() .alias("max"),
                    ])
                )
                all_means [i].append(stats["mean"][0])
                all_stds  [i].append(stats["std"][0])
                all_mins  [i].append(stats["min"][0])
                all_maxs  [i].append(stats["max"][0])
                all_counts[i].append(len(batch_df) * n_levels)

            pbar.update(1)

    # Combine batches per variable
    mean_out, std_out, min_out, max_out = [], [], [], []

    for i in range(n_vars):
        counts = np.array(all_counts[i])
        means  = np.array(all_means[i])
        stds   = np.array(all_stds[i])

        m = (means * counts).sum() / counts.sum()
        ex2 = stds**2 + means**2
        s = np.sqrt(np.clip((ex2 * counts).sum() / counts.sum() - m**2, 0, None))

        mean_out.append(m)
        std_out .append(s)
        min_out .append(np.min(all_mins[i]))
        max_out .append(np.max(all_maxs[i]))

    return mean_out, std_out, min_out, max_out
    
    

## Obtaining statistics

### This is a small dummy version to check the functions

In [7]:
train_ds_dummy = pl.read_csv('/kaggle/input/competitions/leap-atmospheric-physics-ai-climsim/train.csv',
                            n_rows=10)

In [10]:
#------------- Global statistics for variables with vertirtical levels ------------
#------ at each level and variables without vertical levels --------------
#---------- for all rows: space/time coordinates -------------------------------
#----- NOTE: This is only done in the training dataset!

#-- X: features. input variables
mean_X, std_X, min_X, max_X = get_statistics(data_in=train_ds_dummy,
                                             cols_in=FEAT_COLS)
#--Y: target variables. Variables I want to predict
mean_Y, std_Y, min_Y, max_Y = get_statistics(data_in=train_ds_dummy,
                                             cols_in=TARGET_COLS)

#---------- Computing statistcs for vertical variables over the whole -------------
#-------- vertical profile.

#-- X: features. input variables
mean_X_vertical, std_X_vertical, min_X_vertical, max_X_vertical = get_statistics_vertical_var(data_in=train_ds_dummy,
                                                 cols_in=X_col_series ) 
#--Y: target variables. Variables I want to predict
mean_Y_vertical, std_Y_vertical, min_Y_vertical, max_Y_vertical = get_statistics_vertical_var(data_in=train_ds_dummy,
                                                 cols_in=y_col_series ) 

### Running it on the training dataset (365 GB)

#### Benchmarking different chunk size for polars

In [7]:
import polars as pl
import time
import psutil
import os

def benchmark_chunk_size(scan_path, cols_in, n_rows_test, chunk_sizes,parquet):
    """
    Runs get_statistics on a small slice of data for each chunk size,
    reporting time and RAM usage. Use results to pick the best chunk size
    before running on the full 364 GB.
    
    Args:
        scan_path    : path to your CSV file
        cols_in      : list of feature columns (FEAT_COLS)
        n_rows_test  : how many rows to test with (e.g. 10_000)
        chunk_sizes  : list of chunk sizes to benchmark
        parquet      : boolean, to decide if data is parquet or csv
    """
    process = psutil.Process(os.getpid())
    results = []

    for chunk_size in chunk_sizes:
        # --- Set chunk size ---
        pl.Config.set_streaming_chunk_size(chunk_size)

        # --- Measure RAM before ---
        ram_before = process.memory_info().rss / 1e9  # GB

        # --- Run on a small slice ---
        t0 = time.perf_counter()
        if parquet :
            stats = (
            pl.scan_parquet(scan_path)
            .head(n_rows_test)          # <-- only loads n_rows_test rows
            .select(cols_in)
            .select([
                pl.all().mean().name.suffix("_mean"),
                pl.all().std() .name.suffix("_std"),
                pl.all().min() .name.suffix("_min"),
                pl.all().max() .name.suffix("_max"),
            ])
            .collect(engine="streaming")
        )

        else:
            stats = (
                pl.scan_csv(scan_path)
                .head(n_rows_test)          # <-- only loads n_rows_test rows
                .select(cols_in)
                .select([
                    pl.all().mean().name.suffix("_mean"),
                    pl.all().std() .name.suffix("_std"),
                    pl.all().min() .name.suffix("_min"),
                    pl.all().max() .name.suffix("_max"),
                ])
                .collect(engine="streaming")
            )
        elapsed = time.perf_counter() - t0

        # --- Measure RAM after ---
        ram_after  = process.memory_info().rss / 1e9
        ram_delta  = ram_after - ram_before

        results.append({
            "chunk_size" : chunk_size,
            "n_rows"     : n_rows_test,
            "time_s"     : round(elapsed, 3),
            "ram_delta_GB": round(ram_delta, 3),
            "ram_total_GB": round(ram_after, 3),
        })
        print(f"chunk={chunk_size:>7,} | time={elapsed:.3f}s | "
              f"ΔRAM={ram_delta:+.3f} GB | total RAM={ram_after:.3f} GB")

    return pl.DataFrame(results)

In [7]:
#-------- Checking thread count on the Notebook -------
n_threads = pl.thread_pool_size() 
print(n_threads)
#-------- Chunk size used by polars when computing -------
thread_factor = max(12 / n_threads, 1)
n_cols = len(FEAT_COLS)
chunk_size    = max(50_000 / n_cols * thread_factor, 1000)
print(f"Chunk size used automatically by polars: {chunk_size}")
print("This is the number of rows used for each chunk calculation.")

4
Chunk size used automatically by polars: 1000
This is the number of rows used for each chunk calculation.


In [8]:
chunk_sizes  = [1_000, 5_000, 20_000, 50_000, 100_000,500_000]
#chunk_sizes  = [5_000_000]
n_rows_test  = 500_000   # fast to run, enough to see scaling

results_df = benchmark_chunk_size(
    scan_path   = '/kaggle/input/competitions/leap-atmospheric-physics-ai-climsim/train.csv',
    cols_in     = FEAT_COLS,
    n_rows_test = n_rows_test,
    chunk_sizes = chunk_sizes,
    parquet     = False,
)

print(results_df)

chunk=  1,000 | time=116.664s | ΔRAM=+7.474 GB | total RAM=7.612 GB
chunk=  5,000 | time=13.508s | ΔRAM=+1.530 GB | total RAM=9.100 GB
chunk= 20,000 | time=13.594s | ΔRAM=-0.370 GB | total RAM=8.722 GB
chunk= 50,000 | time=13.136s | ΔRAM=-0.059 GB | total RAM=8.643 GB
chunk=100,000 | time=13.125s | ΔRAM=-0.081 GB | total RAM=8.553 GB
chunk=500,000 | time=13.108s | ΔRAM=+0.151 GB | total RAM=8.677 GB
shape: (6, 5)
┌────────────┬────────┬─────────┬──────────────┬──────────────┐
│ chunk_size ┆ n_rows ┆ time_s  ┆ ram_delta_GB ┆ ram_total_GB │
│ ---        ┆ ---    ┆ ---     ┆ ---          ┆ ---          │
│ i64        ┆ i64    ┆ f64     ┆ f64          ┆ f64          │
╞════════════╪════════╪═════════╪══════════════╪══════════════╡
│ 1000       ┆ 500000 ┆ 116.664 ┆ 7.474        ┆ 7.612        │
│ 5000       ┆ 500000 ┆ 13.508  ┆ 1.53         ┆ 9.1          │
│ 20000      ┆ 500000 ┆ 13.594  ┆ -0.37        ┆ 8.722        │
│ 50000      ┆ 500000 ┆ 13.136  ┆ -0.059       ┆ 8.643        │
│ 10000

In [8]:
#----------------- Converting data to parquet --------------
# Run ONCE — this itself will take ~20-40 min but pays back every run
t0 = time.perf_counter()
pl.scan_csv('/kaggle/input/competitions/leap-atmospheric-physics-ai-climsim/train.csv', n_rows=100_000)\
  .sink_parquet(
      '/kaggle/working/train_needed.parquet',
      compression='zstd',
      compression_level=9,   # maximum compression, worth the wait
      row_group_size=100_000
  )
elapsed = time.perf_counter() - t0
print("*"*20, f" DONE, elapsed time: {elapsed} ","*"*20)

********************  DONE, elapsed time: 42.28022032700005  ********************


In [11]:
!ls -ltr

total 508660
-rw-r--r-- 1 root root 520865009 Apr 23 06:56 train_needed.parquet


In [ ]:
#---------------- Benchmarking again but this time using the parquet file ------------------
chunk_sizes  = [1_000, 5_000, 20_000, 50_000, 100_000,500_000]
#chunk_sizes  = [5_000_000]
n_rows_test  = 500_000   # fast to run, enough to see scaling

results_df = benchmark_chunk_size(
    scan_path   = '/kaggle/working/train.parquet',
    cols_in     = FEAT_COLS,
    n_rows_test = n_rows_test,
    chunk_sizes = chunk_sizes,
    parquet     = True,
)

print(results_df)

#### Running in full dataframe

In [20]:
10091520/4

2522880.0

In [8]:
#------------- Global statistics for variables with vertirtical levels ------------
#------ at each level and variables without vertical levels --------------
#---------- for all rows: space/time coordinates -------------------------------
#----- NOTE: This is only done in the training dataset!
pl.Config.set_streaming_chunk_size(100_000)

SLICE_START = 0
SLICE_SIZE  = 2_522_880  # first quarter

#-- X: features. input variables
t0 = time.perf_counter()
mean_X, std_X, min_X, max_X = get_statistics(data_in=train_df,cols_in=FEAT_COLS,
                                             n_rows_total=SLICE_SIZE  ,
                                             batch_size=100_000,
                                             row_offset=SLICE_START)
elapsed = time.perf_counter() - t0
print("*"*20, f" DONE, elapsed time: {elapsed} ","*"*20)

#--Y: target variables. Variables I want to predict
t0 = time.perf_counter()
mean_Y, std_Y, min_Y, max_Y = get_statistics(data_in=train_df,cols_in=TARGET_COLS, 
                                             n_rows_total=SLICE_SIZE  , 
                                             batch_size=100_000,
                                             row_offset=SLICE_START)
elapsed = time.perf_counter() - t0
print("*"*20, f" DONE, elapsed time: {elapsed} ","*"*20)


Computing statistics:   0%|          | 0/26 [00:00<?, ?batch/s]

********************  DONE, elapsed time: 618.0362432909999  ********************


Computing statistics:   0%|          | 0/26 [00:00<?, ?batch/s]

********************  DONE, elapsed time: 558.8894252129999  ********************


In [8]:
mean_X[np.isnan(std_X)]


array([], dtype=float64)

#-------------------------- With all data in one chunk #-------------------


Computing statistics: 


 101/? [46:51<00:00, 29.23s/batch, rows=10,091,520, RAM=35.2 GB]

 
********************  DONE, elapsed time: 2811.90726687  ********************

Computing statistics: 


 101/? [36:26<00:00, 19.86s/batch, rows=10,091,520, RAM=32.6 GB]

 
********************  DONE, elapsed time: 2186.5930735639995  ********************

In [9]:
#---------------------- Saving statistics ---------------------------
#-- X: features. input variables
np.save('/kaggle/working/mean_X_part0.npy', mean_X)
np.save('/kaggle/working/std_X_part0.npy',  std_X)
np.save('/kaggle/working/min_X_part0.npy',  min_X)
np.save('/kaggle/working/max_X_part0.npy',  max_X)

#--Y: target variables. Variables I want to predict
np.save('/kaggle/working/mean_Y_part0.npy', mean_Y)
np.save('/kaggle/working/std_Y_part0.npy',  std_Y)
np.save('/kaggle/working/min_Y_part0.npy',  min_Y)
np.save('/kaggle/working/max_Y_part0.npy',  max_Y)

In [10]:
!ls

max_X_part0.npy  mean_X_part0.npy  min_X_part0.npy  std_X_part0.npy
max_Y_part0.npy  mean_Y_part0.npy  min_Y_part0.npy  std_Y_part0.npy


### Computing statistics for variables over the whole vertical profile

In [11]:
#---------- Computing statistcs for vertical variables over the whole -------------
#-------- vertical profile.
SLICE_START = 0
SLICE_SIZE  = 2_522_880  # first quarter
t0 = time.perf_counter()
#-- X: features. input variables
mean_X_vertical, std_X_vertical, min_X_vertical, max_X_vertical = get_statistics_vertical_var(data_in=train_df,
                                                 cols_in=X_col_series,
                                                 n_rows_total=SLICE_SIZE, 
                                                 batch_size=100_000,
                                                 row_offset=SLICE_START ) 
elapsed = time.perf_counter() - t0
print("*"*20, f" DONE, elapsed time: {elapsed} ","*"*20)
#--Y: target variables. Variables I want to predict
t0 = time.perf_counter()
mean_Y_vertical, std_Y_vertical, min_Y_vertical, max_Y_vertical = get_statistics_vertical_var(data_in=train_df,
                                                 cols_in=y_col_series,
                                                 n_rows_total=SLICE_SIZE, 
                                                 batch_size=100_000,
                                                 row_offset=SLICE_START) 
elapsed = time.perf_counter() - t0
print("*"*20, f" DONE, elapsed time: {elapsed} ","*"*20)

Reading CSV:   0%|          | 0/26 [00:00<?, ?batch/s]

********************  DONE, elapsed time: 578.5197119240001  ********************


Reading CSV:   0%|          | 0/26 [00:00<?, ?batch/s]

********************  DONE, elapsed time: 575.5137011490001  ********************


In [12]:
#---------------------- Saving statistics ---------------------------
#-- X: features. input variables
np.save('/kaggle/working/mean_X_vertical_part0.npy', mean_X_vertical)
np.save('/kaggle/working/std_X_vertical_part0.npy',  std_X_vertical)
np.save('/kaggle/working/min_X_vertical_part0.npy',  min_X_vertical)
np.save('/kaggle/working/max_X_vertical_part0.npy',  max_X_vertical)

#--Y: target variables. Variables I want to predict
np.save('/kaggle/working/mean_Y_vertical_part0.npy', mean_Y_vertical)
np.save('/kaggle/working/std_Y_vertical_part0.npy',  std_Y_vertical)
np.save('/kaggle/working/min_Y_vertical_part0.npy',  min_Y_vertical)
np.save('/kaggle/working/max_Y_vertical_part0.npy',  max_Y_vertical)

In [13]:
!ls

max_X_part0.npy		   mean_Y_part0.npy	      std_X_part0.npy
max_X_vertical_part0.npy   mean_Y_vertical_part0.npy  std_X_vertical_part0.npy
max_Y_part0.npy		   min_X_part0.npy	      std_Y_part0.npy
max_Y_vertical_part0.npy   min_X_vertical_part0.npy   std_Y_vertical_part0.npy
mean_X_part0.npy	   min_Y_part0.npy
mean_X_vertical_part0.npy  min_Y_vertical_part0.npy


---
**End of notebook**